# 09 — ML Readiness & Model Roadmap
**Spacecraft Telemetry Anomaly Detection | Stage 1**

---
**Goal:** Summarise everything accomplished in Stage 1 and map the path forward to model development.

> This section serves as the bridge between the preprocessing pipeline (Stage 1) and  
> anomaly model development (Stage 2 onwards).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os, warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.facecolor':'white', 'axes.facecolor':'#f8f9fa',
                     'axes.titlesize':11, 'font.size':10})
os.makedirs('plots_v2', exist_ok=True)
print('Libraries loaded.')

### 9.1 Stage 1 Pipeline — Completion Summary

In [ ]:
pipeline = pd.DataFrame({
    'Stage': [
        '1. Dataset Loading',
        '2. Data Quality Assessment',
        '3. Exploratory Data Analysis',
        '4. Timestamp Processing',
        '5. Feature Engineering',
        '6. Feature Selection Analysis',
        '7. Long → Wide Transformation',
        '8. Scaling',
    ],
    'What Was Done': [
        '10 000 telemetry rows | 50 parameters | 161 commands loaded and inspected',
        'Zero missing values, duplicates, or timestamp errors — dataset is clean',
        'Distributions, subsystem boxplots, time-series trends — normal baseline characterised',
        '9 temporal context features extracted (hour, weekday, minute_of_day, elapsed_sec, ...)',
        '10 signal features per parameter: rolling stats, lags, z-score, change rate',
        'High-correlation pairs identified, variance analysed, feature sets recommended per model',
        'Wide-format ML matrix: 10 000 rows × 51 cols | forward-fill applied for NaN',
        'StandardScaler (mean=0, std=1) and MinMaxScaler [0,1] — both saved',
    ],
    'Status': ['✓ Complete'] * 8
})
display(pipeline)

# This pipeline has built a NORMAL OPERATIONS baseline.
# The next step is anomaly injection followed by model development.

### 9.2 Candidate Models — Comparison

In [ ]:
models = pd.DataFrame({
    'Model':              ['Isolation Forest','One-Class SVM',
                           'GRU Autoencoder','TCN Autoencoder','NCDE'],
    'Category':           ['Classical','Classical','Deep Learning','Deep Learning','Deep Learning'],
    'Anomaly Mechanism':  [
        'Anomalies are easier to isolate → shorter path in random trees',
        'Learns a tight boundary around normal space; anomalies fall outside it',
        'Learns to reconstruct normal sequences; anomalies have high reconstruction error',
        'Same as GRU but uses dilated causal convolutions — more efficient',
        'Fits a continuous ODE to normal trajectory; anomaly = trajectory deviation',
    ],
    'Temporal Aware':     ['No','No','Yes','Yes','Yes (continuous)'],
    'Input':              [
        'Standard-scaled wide matrix',
        'Standard-scaled wide matrix',
        'MinMax-scaled sequences',
        'MinMax-scaled sequences',
        'Long format + timestamps',
    ],
    'Stage':              ['Stage 2 Baseline','Stage 2 Baseline','Stage 3','Stage 3','Stage 4'],
})
display(models)

# KEY NOTE:
# Stage 2 (classical) models use the WIDE format output (Section 07 + 08)
# Stage 3 (deep learning) models will use sliding-window sequences over time
# Stage 4 (NCDE) will use the original LONG format — it handles irregular sampling natively

### 9.3 Development Roadmap Visualisation

In [ ]:
stages = [
    ('Stage 1\n(Current)', 'Data Engineering\n& Preprocessing\n[DONE]', '#2ca02c'),
    ('Stage 2',            'Anomaly Injection\n+ Isolation Forest\n& One-Class SVM', '#1f77b4'),
    ('Stage 3',            'GRU Autoencoder\n& TCN Autoencoder\n(Deep Learning)', '#9467bd'),
    ('Stage 4',            'NCDE\n(Advanced\nIrregular TS)', '#d62728'),
]

fig, ax = plt.subplots(figsize=(16, 4))
ax.set_xlim(-0.5, len(stages) - 0.5)
ax.set_ylim(-0.6, 1.2)
ax.axis('off')
ax.set_title('Spacecraft Telemetry Anomaly Detection — Development Roadmap',
             fontsize=13, fontweight='bold', pad=12)

for i, (stage, desc, color) in enumerate(stages):
    rect = mpatches.FancyBboxPatch(
        (i - 0.42, -0.45), 0.84, 1.35,
        boxstyle='round,pad=0.05',
        facecolor=color, edgecolor='white',
        alpha=0.85, linewidth=2
    )
    ax.add_patch(rect)
    ax.text(i, 0.75, stage, ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')
    ax.text(i, 0.05, desc, ha='center', va='center',
            fontsize=8, color='white')
    if i < len(stages) - 1:
        ax.annotate('', xy=(i + 0.5, 0.3), xytext=(i + 0.42, 0.3),
                    arrowprops=dict(arrowstyle='->', color='#495057', lw=2))

plt.tight_layout()
plt.savefig('plots_v2/09_roadmap.png', dpi=150, bbox_inches='tight')
plt.show()

# The green Stage 1 box (current) feeds all downstream models
# Each subsequent stage builds on the preprocessed data produced here

### 9.4 All Output Files — Ready for Stage 2

In [ ]:
file_registry = [
    # (filename, description, next used in)
    ('telemetry_train.csv',
     'Raw telemetry — long format, 10 000 rows',
     'Stage 4 NCDE (long format model)'),
    ('telecommand_train.csv',
     'Raw telecommand — 161 commands, 96% success rate',
     'Stage 2 contextual analysis'),
    ('processed_v2/telemetry_temporal.csv',
     'Telemetry + 9 temporal context features',
     'Feature analysis reference'),
    ('processed_v2/telemetry_engineered.csv',
     'Telemetry + all 10 engineered signal features (22 cols total)',
     'Stage 3 DL model input preparation'),
    ('processed_v2/telemetry_wide.csv',
     'Wide-format: 10 000 rows × 51 cols | forward-filled',
     'Stage 2 classical ML direct input'),
    ('processed_v2/telemetry_standard_scaled.csv',
     'Wide + StandardScaler applied → Isolation Forest / OCSVM input',
     'Stage 2 — primary baseline model input'),
    ('processed_v2/telemetry_minmax_scaled.csv',
     'Wide + MinMaxScaler [0,1] → GRU / TCN Autoencoder input',
     'Stage 3 — deep learning model input'),
]

reg_df = pd.DataFrame(file_registry, columns=['File', 'Description', 'Next Used In'])

# Show file sizes
def get_size(f):
    try:
        return f'{os.path.getsize(f):,} bytes'
    except:
        return 'Not found'

import os
reg_df['Size'] = reg_df['File'].apply(get_size)
display(reg_df)

print('\n' + '='*70)
print('STAGE 1 COMPLETE — All preprocessing outputs validated and saved.')
print('Next: Anomaly injection + Stage 2 model development.')
print('='*70)